# Segmentation alpha: view the crop

A 500 × 500 µm square at full resolution (0.325 µm/px), centred on T3 of Mouse 2 `slide04_s3`. It's made by `make_crop.py` in this folder; `mouse02_slide04_s3_T3_500um.json` records where it came from.

In [ ]:
%gui qt
from pathlib import Path

import napari
import numpy as np
import tifffile

CROP = Path("mouse02_slide04_s3_T3_500um.ome.tif")

with tifffile.TiffFile(CROP) as tf:
    image = tf.asarray()
    meta = tf.ome_metadata
px_um = float(meta.split('PhysicalSizeX="', 1)[1].split('"', 1)[0])
channels = [c.split('Name="', 1)[1].split('"', 1)[0] for c in meta.split("<Channel ")[1:]]
print(CROP.name, image.shape, image.dtype, f"{px_um} µm/px", channels)

In [ ]:
STYLE = {  # name, colormap, visible, opacity
    "CY5":   ("NeuN (CY5)", "gray", True, 0.4),
    "TRITC": ("GFP stain (TRITC)", "green", True, 1.0),
    "DAPI":  ("DAPI", "blue", True, 1.0),
    "FITC":  ("native GFP (FITC)", "yellow", False, 1.0),
}

viewer = napari.Viewer(title=CROP.stem)
for i, ch in enumerate(channels):
    name, cmap, visible, opacity = STYLE.get(ch, (ch, "gray", False, 1.0))
    viewer.add_image(image[i], name=name, colormap=cmap, blending="additive", visible=visible,
                     opacity=opacity, contrast_limits=tuple(np.percentile(image[i], [1, 99.8])),
                     scale=(px_um, px_um), units="um")
viewer.canvas.overlays.scale_bar.visible = True